# 🔢 Train License Plate Detection + OCR

> **Đề tài:** Hệ thống Giám sát Phương tiện Giao thông – DATN_VTHUW  
> **Model 1:** YOLOv8n – Phát hiện vùng biển số (detect bounding box)  
> **Model 2:** PaddleOCR – Đọc ký tự từ vùng biển số  
> **Dataset:** Vietnamese License Plate datasets (Roboflow + GitHub)  
> **Output:** `license_plate_detection.pt` → lưu Google Drive

⚡ **Bật GPU:** Runtime → Change runtime type → **T4 GPU**

---
### Phương pháp 2 bước:
```
Frame video
    ↓
YOLOv8n (detect vùng biển số)
    ↓
Crop vùng biển số
    ↓
PaddleOCR (đọc ký tự) → '51F1-23456'
```

## 📦 Bước 1: Cài đặt

In [ ]:
!nvidia-smi
!pip install ultralytics roboflow --quiet

import ultralytics, torch
ultralytics.checks()
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

## ☁️ Bước 2: Google Drive

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/DATN_TrafficAI'
MODEL_DIR = f'{DRIVE_DIR}/models'
LOG_DIR   = f'{DRIVE_DIR}/logs'

for d in [MODEL_DIR, LOG_DIR]:
    os.makedirs(d, exist_ok=True)

print(f'✅ Drive ready. Models: {MODEL_DIR}')

## 📂 Bước 3: Tải Dataset Biển số xe

> Ưu tiên dataset biển số **Việt Nam** vì format biển số khác quốc tế

In [ ]:
# ==============================================================
# 🅰️ CÁCH A: Roboflow – License Plate Detection (khuyến nghị)
# ==============================================================

RF_API_KEY = 'YOUR_ROBOFLOW_API_KEY'  # <-- THAY VÀO ĐÂY

# Các dataset biển số có chất lượng tốt:
PLATE_DATASETS = {
    # Option 1: License Plate Detection tổng hợp (6,000+ ảnh)
    'general': {
        'workspace': 'augmented-startups',
        'project': 'vehicle-registration-plates-trudk',
        'version': 1,
    },
    # Option 2: Vietnamese License Plate (ảnh thực tế VN)
    'vietnam': {
        'workspace': 'license-plate-detection-hv4du',
        'project': 'license-plate-detection-dxapu',
        'version': 1,
    },
    # Option 3: License Plate Dataset (đa dạng châu Á)
    'asia': {
        'workspace': 'roboflow-universe-projects',
        'project': 'license-plate-recognition-rxg4e',
        'version': 4,
    },
}

CHOSEN = 'general'  # <-- Đổi: 'general', 'vietnam', 'asia'

from roboflow import Roboflow
rf = Roboflow(api_key=RF_API_KEY)

cfg = PLATE_DATASETS[CHOSEN]
project = rf.workspace(cfg['workspace']).project(cfg['project'])
dataset = project.version(cfg['version']).download('yolov8')

DATASET_PATH = dataset.location
print(f'✅ Dataset [{CHOSEN}]: {DATASET_PATH}')

In [ ]:
# ==============================================================
# 🅱️ CÁCH B: GitHub – Vietnamese License Plate Dataset (miễn phí)
# Nguồn: nhóm nghiên cứu ANPR Việt Nam
# ==============================================================

# !git clone https://github.com/nicehorse06/license-plate-detector.git /content/lp_github

# Hoặc dùng dataset từ link Google Drive công khai:
# !gdown 'GOOGLE_DRIVE_FILE_ID' -O /content/datasets/license_plate.zip
# !unzip -q /content/datasets/license_plate.zip -d /content/datasets/

# DATASET_PATH = '/content/datasets/license_plate'

print('Option B: Bỏ comment và thay file ID từ Google Drive')

In [ ]:
# ==============================================================
# 🅲 CÁCH C: Tạo dataset từ video/ảnh giao thông Việt Nam
# Script tự động crop biển số từ video
# ==============================================================

# Bước 1: Upload video lên Drive
# Bước 2: Dùng YOLOv8 pretrained để detect và crop biển số
# Bước 3: Gán nhãn manual bằng Roboflow hoặc LabelImg

# !pip install labelImg  # GUI annotation tool
# Script tạo dataset từ video:

GENERATE_FROM_VIDEO = False  # Đổi True để chạy

if GENERATE_FROM_VIDEO:
    import cv2
    from ultralytics import YOLO

    # Dùng model pretrained để extract biển số
    model = YOLO('yolov8n.pt')

    VIDEO_PATH = '/content/drive/MyDrive/DATN_TrafficAI/videos/traffic.mp4'
    cap = cv2.VideoCapture(VIDEO_PATH)

    os.makedirs('/content/plate_crops', exist_ok=True)
    frame_id = 0
    saved = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break

        if frame_id % 5 == 0:  # Lấy 1 frame mỗi 5 frame
            results = model(frame, classes=[2, 3, 5, 7], conf=0.5)  # cars, trucks
            for r in results:
                for box in r.boxes:
                    x1, y1, x2, y2 = map(int, box.xyxy[0])
                    crop = frame[y1:y2, x1:x2]
                    if crop.size > 0:
                        cv2.imwrite(f'/content/plate_crops/frame_{frame_id}_{saved}.jpg', crop)
                        saved += 1

        frame_id += 1
        if frame_id > 1000: break

    cap.release()
    print(f'Đã extract {saved} crops để gán nhãn')

In [ ]:
# Xem nội dung data.yaml
import glob, yaml

yaml_files = glob.glob(f'{DATASET_PATH}/**/*.yaml', recursive=True)
YAML_FILE = yaml_files[0] if yaml_files else None

if YAML_FILE:
    with open(YAML_FILE) as f:
        cfg = yaml.safe_load(f)

    print(f'Classes: {cfg.get("names", [])}')
    print(f'NC: {cfg.get("nc", 0)}')

    # Đảm bảo chỉ có 1 class: license_plate
    cfg['names'] = ['license_plate']
    cfg['nc'] = 1
    cfg['path'] = DATASET_PATH
    cfg['train'] = 'train/images'
    cfg['val'] = 'valid/images'

    with open(YAML_FILE, 'w') as f:
        yaml.dump(cfg, f, default_flow_style=False)

    n_train = len(glob.glob(f'{DATASET_PATH}/train/images/*'))
    n_val   = len(glob.glob(f'{DATASET_PATH}/valid/images/*'))
    print(f'\nTrain: {n_train} | Val: {n_val}')
    print('✅ data.yaml đã cập nhật')

## 🚀 Bước 4: Huấn luyện YOLOv8 Plate Detector

In [ ]:
from ultralytics import YOLO

# YOLOv8n đủ tốt cho bài toán detect 1 class (license_plate)
# Nhẹ hơn, chạy nhanh hơn trong realtime
model = YOLO('yolov8n.pt')

results = model.train(
    data=YAML_FILE,
    epochs=60,
    imgsz=640,
    batch=16,
    device=0,
    patience=20,
    name='license_plate_detection',
    project='/content/runs',
    save=True,
    save_period=10,
    exist_ok=True,
    amp=True,
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    # Augmentation cho license plate
    fliplr=0.5,
    flipud=0.0,
    mosaic=0.5,           # Ít mosaic hơn vì biển số nhỏ
    mixup=0.0,
    degrees=5.0,          # Xoay nhẹ
    translate=0.1,
    scale=0.4,            # Scale range
    perspective=0.0005,   # Perspective transform (góc chụp khác nhau)
    hsv_s=0.7,
    hsv_v=0.4,
)

print('\n✅ License Plate Detection training xong!')

## 📊 Bước 5: Đánh giá

In [ ]:
from IPython.display import Image as IPImage, display
import os

run_dir = '/content/runs/license_plate_detection'

for img in ['results.png', 'confusion_matrix_normalized.png', 'PR_curve.png']:
    path = f'{run_dir}/{img}'
    if os.path.exists(path):
        print(f'\n--- {img} ---')
        display(IPImage(path, width=800))

best_model = YOLO(f'{run_dir}/weights/best.pt')
metrics = best_model.val()
print(f'\nmAP@50: {metrics.box.map50:.4f}')
print(f'mAP@50-95: {metrics.box.map:.4f}')

## 🔤 Bước 6: Test OCR Pipeline (YOLOv8 + PaddleOCR)

> Pipeline hoàn chỉnh: Detect biển số → Crop → Đọc ký tự

In [ ]:
# Cài PaddleOCR
!pip install paddlepaddle-gpu paddleocr --quiet
# Nếu không có GPU:
# !pip install paddlepaddle paddleocr --quiet

In [ ]:
import cv2
import numpy as np
import glob
from ultralytics import YOLO
from paddleocr import PaddleOCR
from IPython.display import Image as IPImage, display
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# Load models
plate_model = YOLO(f'/content/runs/license_plate_detection/weights/best.pt')

# PaddleOCR với tiếng Anh (biển số VN dùng chữ cái Latin)
ocr = PaddleOCR(
    use_angle_cls=True,
    lang='en',           # Biển số VN dùng tiếng Anh
    use_gpu=True,
    show_log=False,
)

def detect_and_read_plate(image_path):
    """Pipeline: detect biển số → crop → OCR"""
    img = cv2.imread(image_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Detect biển số
    results = plate_model(img, conf=0.4)
    detections = []

    fig, ax = plt.subplots(1, 1, figsize=(12, 6))
    ax.imshow(img_rgb)

    for r in results:
        for box in r.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            conf = float(box.conf[0])

            # Crop vùng biển số
            crop = img[y1:y2, x1:x2]

            # Tiền xử lý ảnh để OCR chính xác hơn
            h, w = crop.shape[:2]
            if w < 100: crop = cv2.resize(crop, (200, int(200*h/w)))
            gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
            _, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
            enhanced = cv2.cvtColor(thresh, cv2.COLOR_GRAY2RGB)

            # OCR
            ocr_result = ocr.ocr(enhanced, cls=True)
            texts = []
            if ocr_result and ocr_result[0]:
                for line in ocr_result[0]:
                    text, conf_ocr = line[1]
                    if conf_ocr > 0.5:
                        texts.append(text)

            plate_text = ' '.join(texts).upper().strip()
            detections.append({'bbox': (x1, y1, x2, y2), 'conf': conf, 'text': plate_text})

            # Vẽ kết quả
            rect = patches.Rectangle((x1, y1), x2-x1, y2-y1,
                                     linewidth=2, edgecolor='lime', facecolor='none')
            ax.add_patch(rect)
            ax.text(x1, y1-5, f'{plate_text} ({conf:.2f})',
                   color='lime', fontsize=12, fontweight='bold',
                   bbox=dict(boxstyle='round,pad=0.2', facecolor='black', alpha=0.7))

    ax.set_title(f'License Plate Detection + OCR', fontsize=14)
    ax.axis('off')
    plt.tight_layout()
    plt.show()

    return detections


# Test với ảnh val
val_imgs = glob.glob(f'{DATASET_PATH}/valid/images/*')[:3]
for img_path in val_imgs:
    print(f'\n🔍 Xử lý: {os.path.basename(img_path)}')
    dets = detect_and_read_plate(img_path)
    for d in dets:
        print(f'  Biển số: "{d["text"]}" (conf: {d["conf"]:.2f})')

## 💾 Bước 7: Lưu tất cả Models

In [ ]:
import shutil

# Lưu YOLO plate detector
best_pt = '/content/runs/license_plate_detection/weights/best.pt'
dest    = f'{MODEL_DIR}/license_plate_detection.pt'
shutil.copy2(best_pt, dest)
print(f'✅ Plate model: {dest} ({os.path.getsize(dest)/1e6:.1f} MB)')

# Lưu logs
shutil.copytree('/content/runs/license_plate_detection',
                f'{LOG_DIR}/license_plate_detection', dirs_exist_ok=True)

# Liệt kê tất cả models trên Drive
print('\n📁 Tất cả models trên Drive:')
for f in os.listdir(MODEL_DIR):
    size = os.path.getsize(f'{MODEL_DIR}/{f}') / 1e6
    print(f'   {f}: {size:.1f} MB')

print('\n🎉 Hoàn tất!')
print('Tải về máy: Google Drive → DATN_TrafficAI/models/')
print('Copy vào:   DATN_VTHUW/models/')

## 📋 Bước 8: Export sang ONNX (Tùy chọn)

> ONNX giúp model chạy nhanh hơn 20-30% khi inference

In [ ]:
# Export plate model sang ONNX
plate_model_export = YOLO(f'{MODEL_DIR}/license_plate_detection.pt')
plate_model_export.export(format='onnx', dynamic=True, simplify=True)

onnx_path = f'/content/runs/license_plate_detection/weights/best.onnx'
onnx_dest = f'{MODEL_DIR}/license_plate_detection.onnx'
shutil.copy2(onnx_path, onnx_dest)
print(f'✅ ONNX model: {onnx_dest}')